In [ ]:
# ensemble : stack 
import os
from glob import glob
from tqdm import tqdm
import rasterio

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset, Subset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchmetrics.functional import structural_similarity_index_measure as ssim_index
from transformers import SegformerConfig, SegformerForSemanticSegmentation


import re
import numpy as np
import pandas as pd

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'retina'

import warnings
from rasterio.errors import NotGeoreferencedWarning
warnings.simplefilter("ignore", NotGeoreferencedWarning)
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=r"Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated.*"
)


In [ ]:
# device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(42)
print(device)

In [ ]:
# directories
himachal_raster = 'datasets128/himachal_128/images'
himachal_mask = 'datasets128/himachal_128/masks' 

himlad_raster = 'datasets128/himachal_ladakh_128/images'
himlad_mask = 'datasets128/himachal_ladakh_128/masks'

sikkim_raster = 'datasets128/sikkim128/images'
sikkim_mask = 'datasets128/sikkim128/masks'

kashmir_raster = 'datasets128/kashmir128/images'
kashmir_mask = 'datasets128/kashmir128/masks'

uttrakhand_raster = 'datasets128/uttrakhand128/images'
uttrakhand_mask = 'datasets128/uttrakhand128/masks'

#loss function : focal + dice + tversky

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, y_pred, y_true):
        eps = 1e-7
        y_pred = torch.clamp(y_pred, eps, 1. - eps)  # prevent log(0)
        bce = - (y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred))
        pt = torch.where(y_true == 1, y_pred, 1 - y_pred)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce
        return focal_loss.mean()
        
class DiceLoss(nn.Module):
    def __init__(self):
        super(DiceLoss, self).__init__()

    def forward(self, y_pred, y_true):
        eps = 1e-7
        y_pred = y_pred.squeeze(1)
        y_true = y_true.squeeze(1)
        intersection = (y_pred * y_true).sum()
        union = y_pred.sum() + y_true.sum()
        dice = (2. * intersection + eps) / (union + eps)
        return 1 - dice
        
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3):
        super(TverskyLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, y_pred, y_true):
        eps = 1e-7
        y_pred = y_pred.squeeze(1)
        y_true = y_true.squeeze(1)
        TP = (y_pred * y_true).sum()
        FP = ((1 - y_true) * y_pred).sum()
        FN = (y_true * (1 - y_pred)).sum()
        tversky = (TP + eps) / (TP + self.alpha * FP + self.beta * FN + eps)
        return 1 - tversky
        
class CombinedLoss(nn.Module):
    def __init__(self, weights=(1.0, 0.5, 0.5)):
        super(CombinedLoss, self).__init__()
        self.focal = FocalLoss()
        self.dice = DiceLoss()
        self.tversky = TverskyLoss()
        self.w_focal, self.w_dice, self.w_tversky = weights

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)  # ✅ for binary segmentation
        loss = (
            self.w_focal * self.focal(probs, targets.float()) +
            self.w_dice * self.dice(probs, targets.float()) +
            self.w_tversky * self.tversky(probs, targets.float())
        )
        return loss

focal_dice_tversky = CombinedLoss(weights=(1.0, 0.5, 0.5))

In [ ]:

#-----------------------ResUNet-------------------------------
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        ) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return F.relu(out)

class ResUNet(nn.Module):
    def __init__(self, in_channels=18, out_channels=1):
        super().__init__()
        self.init_conv = nn.Conv2d(in_channels, 64, kernel_size=3, padding=1)
        self.enc1 = ResidualBlock(64, 64)
        self.enc2 = ResidualBlock(64, 128)
        self.enc3 = ResidualBlock(128, 256)
        self.enc4 = ResidualBlock(256, 512)
        self.bottleneck = ResidualBlock(512, 1024)
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec1 = ResidualBlock(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec2 = ResidualBlock(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ResidualBlock(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec4 = ResidualBlock(128, 64)
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.enc1(F.relu(self.init_conv(x)))
        x2 = self.enc2(F.max_pool2d(x1, 2))
        x3 = self.enc3(F.max_pool2d(x2, 2))
        x4 = self.enc4(F.max_pool2d(x3, 2))
        x5 = self.bottleneck(F.max_pool2d(x4, 2))
        x = self.up1(x5)
        x = self.dec1(torch.cat([x, x4], dim=1))
        x = self.up2(x)
        x = self.dec2(torch.cat([x, x3], dim=1))
        x = self.up3(x)
        x = self.dec3(torch.cat([x, x2], dim=1))
        x = self.up4(x)
        x = self.dec4(torch.cat([x, x1], dim=1))
        return self.final_conv(x)
    

In [ ]:
#-----------------------SegformerB4-------------------------------
config = SegformerConfig.from_pretrained("nvidia/segformer-b4-finetuned-ade-512-512")
config.num_channels = 18   # your 18-band input
config.num_labels = 1      # glacier vs background
config.image_size = 128 
segformerb4 = SegformerForSemanticSegmentation(config)

# from torchinfo import summary
# summary(segformerb4, input_size=(1,18,128,128))

In [ ]:
# importing model weights

resunet = ResUNet(in_channels = 18, out_channels = 1)
resunet.to(device)
resunet.load_state_dict(torch.load("Github weights/ResUNet(focalDiceTversky).pt"))
resunet.eval()

segformerb4.to(device)
segformerb4.load_state_dict(torch.load("Github weights/SegformerB4(focalDiceTversky).pt"))
segformerb4.eval()

In [ ]:
def stacked_output(x):
    x = x.to(device)
    with torch.no_grad():
        out1 = torch.sigmoid(resunet(x))
        out2 = torch.sigmoid(segformerb4(x).logits)
        stacked = torch.cat((out1, out2), dim=1)  # shape: (batch_size, 2, H, W)
    return stacked